# Insurance Website Scraping Example

This notebook demonstrates how to use the scraping module to extract clean markdown content from insurance websites.

Each insurance provider has **one URL** that gets scraped and saved as `webpage_{timestamp}.md`

In [1]:
import sys
sys.path.append('..')

import asyncio
from src.scraping import InsuranceScraper, LLMContentParser
from src.scraping.config import get_all_configs, get_provider_config

## 1. View Configured Providers

Let's see which insurance providers are configured and which have URLs ready.

In [2]:
# Get all configured insurance providers
all_configs = get_all_configs()

print(f"📋 Total providers configured: {len(all_configs)}\n")
print("="*80 + "\n")

configured_count = 0
for config in all_configs:
    status = "✅" if config.url else "⚠️"
    print(f"{status} {config.provider_name} ({config.provider_slug})")
    
    if config.url:
        configured_count += 1
        print(f"   🔗 {config.url}")
    else:
        print(f"   ⚠️  No URL configured yet")
    
    if config.description:
        print(f"   📝 {config.description}")
    print()

print("="*80)
print(f"\n✅ {configured_count}/{len(all_configs)} providers have URLs configured")

📋 Total providers configured: 16


✅ Goudse Expat Pakket (goudse_expat_pakket)
   🔗 https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/de-goudse-verzekeringen/goudse-expat-pakket/
   📝 Goudse expat insurance package

✅ Goudse NGO Zendelingen (goudse_ngo_zendelingen)
   🔗 https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/de-goudse-verzekeringen/ngo/
   📝 NGO and missionary insurance coverage

✅ Goudse Working Nomad (goudse_working_nomad)
   🔗 https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/de-goudse-verzekeringen/isis-working-nomad-reisverzekering/
   📝 Working nomad insurance

✅ Allianz Care (allianz_care)
   🔗 https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/allianz-care/international-health-insurance/
   📝 Allianz international health insurance

✅ Allianz Globetrotter (allianz_globetrotter)
   🔗 https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/allianz-global-assi

## 2. Test LLM Parser

Test the LLM-based HTML to Markdown parser with sample HTML.

In [3]:
# Initialize parser
parser = LLMContentParser()

# Sample HTML (replace with real HTML)
sample_html = """
<html>
<body>
    <nav>Navigation menu...</nav>
    <main>
        <h1>Insurance Coverage</h1>
        <p>Our premium plan includes comprehensive medical coverage.</p>
        <h2>Benefits</h2>
        <ul>
            <li>Medical expenses up to €1,000,000</li>
            <li>Emergency evacuation</li>
            <li>Dental coverage up to €5,000</li>
        </ul>
    </main>
    <footer>Footer content...</footer>
</body>
</html>
"""

# Parse HTML to markdown
markdown = parser.parse_html_to_markdown(
    html_content=sample_html,
    url="https://example.com/test"
)

print("Extracted Markdown:\n")
print(markdown)

Extracted Markdown:

# Insurance Coverage

Our premium plan includes comprehensive medical coverage.

## Benefits

*   Medical expenses up to €1,000,000
*   Emergency evacuation
*   Dental coverage up to €5,000


## 3. Scrape a Single Provider

Scrape the webpage for a specific insurance provider.

In [4]:
# Initialize scraper
scraper = InsuranceScraper()

# Choose a provider to scrape
provider_slug = "goudse_expat_pakket"

# Check if URL is configured
config = get_provider_config(provider_slug)

if not config or not config.url:
    print(f"⚠️  No URL configured for {provider_slug}")
    print("Add URL to src/scraping/config.py before scraping")
else:
    print(f"📍 Provider: {config.provider_name}")
    print(f"🔗 URL: {config.url}")
    print(f"\nScraping {provider_slug}...\n")
    
    # Run scraper (save_to_disk=False for testing)
    url, content = await scraper.scrape_provider(
        provider_slug=provider_slug,
        save_to_disk=False  # Set to True to save files
    )
    
    # Display result
    print(f"\nResult for {provider_slug}:")
    if content:
        print(f"  ✅ Successfully scraped {len(content)} characters")
        print(f"\n📄 Preview (first 1000 chars):\n")
        print(content[:1000])
        print("\n...")
    else:
        print(f"  ❌ Scraping failed")

📍 Provider: Goudse Expat Pakket
🔗 URL: https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/de-goudse-verzekeringen/goudse-expat-pakket/

Scraping goudse_expat_pakket...



[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.34s 

[SCRAPE].. ◆ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.37s 


Result for goudse_expat_pakket:
  ✅ Successfully scraped 16677 characters

📄 Preview (first 1000 chars):

# Goudse Expat Package

The Goudse Expat Package is a comprehensive international insurance package specifically designed for Dutch and Belgian citizens who emigrate abroad to live and work, or who are already residing abroad.

This package offers extensive coverage for healthcare costs, including coverage in private clinics and single rooms (Excellent coverage level) and medical repatriation. Additionally, the Goudse Expat Package includes various other important insurances such as travel, travel baggage, cancellation, disability, liability, household contents, legal assistance, and accidents.

This expat insurance from De Goudse is a flexible, continuous insurance, specially designed for Dutch and Belgian citizens working abroad long-term. It is suitable if you are seconded by a (foreign) employer, have found local work, or live abroad as a freelancer or self-employed entreprene

## 4. Real Example - Scrape and Save

Scrape a real provider and save to disk.

In [5]:
# Scrape Goudse Expat Pakket and save to disk
scraper = InsuranceScraper()

# Check what we're about to scrape
config = get_provider_config("goudse_expat_pakket")
print(f"📍 Provider: {config.provider_name}")
print(f"🔗 URL: {config.url}")
print(f"📝 Description: {config.description}")
print(f"💾 Will save to: data/documents/goudse_expat_pakket/webpage_YYYYMMDD.md")
print("\n" + "="*60 + "\n")

# Uncomment to actually scrape (uses Gemini API credits):
print("🚀 Starting scrape...")
url, content = await scraper.scrape_provider(
    provider_slug="goudse_expat_pakket",
    save_to_disk=True  # Will save the file
)

if content:
    print(f"\n✅ Successfully scraped!")
    print(f"   Length: {len(content)} characters")
    print(f"\n📄 Preview (first 1000 chars):\n")
    print(content[:1000])
    print("\n...")
else:
    print(f"\n❌ Scraping failed")

📍 Provider: Goudse Expat Pakket
🔗 URL: https://www.expatverzekering.nl/internationale-verzekeringen/verzekeraars/de-goudse-verzekeringen/goudse-expat-pakket/
📝 Description: Goudse expat insurance package
💾 Will save to: data/documents/goudse_expat_pakket/webpage_YYYYMMDD.md


🚀 Starting scrape...


[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.44s 

[SCRAPE].. ◆ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.46s 


✅ Successfully scraped!
   Length: 13794 characters

📄 Preview (first 1000 chars):

# Goudse Expat Package

The Goudse Expat Package is a comprehensive international insurance package designed for Dutch and Belgian citizens who are emigrating or already residing abroad for work or living.

This package offers extensive coverage for healthcare costs, including access to private clinics and single rooms (Excellent coverage level), and medical repatriation. It also includes other essential insurances such as travel, travel baggage, cancellation, disability, liability, household contents, legal assistance, and accidents.

This flexible, continuous expat insurance from De Goudse is specifically for Dutch and Belgians working abroad long-term, whether seconded by an employer, locally employed, or self-employed. It can be applied for by individuals, their family members, or by Dutch employers for their employees abroad.

## 1. Characteristics of Goudse Expat Package

### Policy Duration
*   

## 5. Batch Scraping

Scrape multiple specific providers at once.

In [ ]:
# Define providers to scrape
providers_to_scrape = [
    "goudse_expat_pakket",
    "goudse_ngo_zendelingen",
    "allianz_care"
]

scraper = InsuranceScraper()

# Preview what we'll scrape
print("📋 Providers to scrape:\n")
for slug in providers_to_scrape:
    config = get_provider_config(slug)
    if config and config.url:
        print(f"  ✅ {config.provider_name}")
        print(f"     {config.url}")
    else:
        print(f"  ⚠️  {slug} - No URL configured")
    print()

# Uncomment to actually scrape:
# print("\n🚀 Starting batch scrape...\n")
# results = {}
# 
# for slug in providers_to_scrape:
#     print(f"Scraping {slug}...")
#     url, content = await scraper.scrape_provider(slug, save_to_disk=True)
#     results[slug] = (url, content)
#     if content:
#         print(f"  ✅ Success ({len(content)} chars)\n")
#     else:
#         print(f"  ❌ Failed\n")
# 
# # Summary
# print("\n" + "="*60)
# print("📊 Batch Scraping Summary:\n")
# successful = sum(1 for _, content in results.values() if content)
# print(f"  ✅ Successful: {successful}/{len(results)}")
# print(f"  ❌ Failed: {len(results) - successful}/{len(results)}")

## 6. Scrape All Providers

Scrape all configured insurance providers at once.

**WARNING:** This will use significant API credits!

In [6]:
# Scrape all providers
# WARNING: This can take a while and use API credits!

# Check how many will be scraped
all_configs = get_all_configs()
configured_count = sum(1 for c in all_configs if c.url)
print(f"⚠️  This will scrape {configured_count} insurance provider webpages")
print(f"⚠️  Estimated API cost: ~${configured_count * 0.10:.2f} (rough estimate)")
print(f"⚠️  Estimated time: ~{configured_count * 30} seconds\n")

# Uncomment to run:
scraper = InsuranceScraper()
all_results = await scraper.scrape_all_providers(save_to_disk=True)

print("\n📊 Summary:")
for provider_slug, (url, content) in all_results.items():
    if content:
        print(f"  ✅ {provider_slug}: {len(content)} chars")
    else:
        print(f"  ❌ {provider_slug}: Failed")

⚠️  This will scrape 15 insurance provider webpages
⚠️  Estimated API cost: ~$1.50 (rough estimate)
⚠️  Estimated time: ~450 seconds



[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.31s 

[SCRAPE].. ◆ https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://www.expatverzekering.nl/internationale-v...ars/de-goudse-verzekeringen/goudse-expat-pakket/  |
✓ | ⏱: 1.33s 

...

[FETCH]... ↓ https://www.expatverzekering.nl/internationale-v...rzekeraars/henner/international-expat-insurance/  |
✓ | ⏱: 1.10s 

[SCRAPE].. ◆ https://www.expatverzekering.nl/internationale-v...rzekeraars/henner/international-expat-insurance/  |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://www.expatverzekering.nl/internationale-v...rzekeraars/henner/international-expat-insurance/  |
✓ | ⏱: 1.11s 

No URL configured for provider: MSH



📊 Summary:
  ✅ goudse_expat_pakket: 16717 chars
  ✅ goudse_ngo_zendelingen: 6215 chars
  ✅ goudse_working_nomad: 8754 chars
  ✅ allianz_care: 6464 chars
  ✅ allianz_globetrotter: 3360 chars
  ✅ cigna_close_care: 2482 chars
  ✅ cigna_global_care: 7943 chars
  ✅ expatriate_group: 3046 chars
  ✅ globality_yougenio: 4247 chars
  ✅ oom_tib: 12706 chars
  ✅ oom_wib: 13504 chars
  ✅ special_isis: 3116 chars
  ✅ ACS: 3119 chars
  ✅ IMG_: 6116 chars
  ✅ International Expat Insurance: 13592 chars
  ❌ MSH: Failed


## 7. Verify Saved Files

Check the scraped markdown files.

In [ ]:
from pathlib import Path
from src.config import DOCUMENTS_DIR

# List all scraped webpage files
print("📂 Scraped webpage files:\n")

webpage_files = list(DOCUMENTS_DIR.rglob("webpage_*.md"))

if not webpage_files:
    print("  No scraped files found yet.")
    print("  Run the scraping cells above to create files.")
else:
    for file_path in sorted(webpage_files):
        provider = file_path.parent.name
        size = file_path.stat().st_size
        print(f"  📄 {provider}/{file_path.name} ({size:,} bytes)")
    
    print(f"\n✅ Total files: {len(webpage_files)}")

## Next Steps

1. **Test with one provider** - Run cell 3 or 4 to scrape Goudse Expat Pakket
2. **Batch scrape** - Use cell 5 to scrape multiple providers
3. **Full scrape** - Use the CLI script: `python scripts/scrape_insurance_data.py`
4. **Re-ingest documents** - After scraping: `python scripts/ingest_documents.py --recreate`
5. **Set up monthly automation** - Use `scripts/monthly_update.sh`

### Files are saved to:
```
data/documents/{provider_slug}/webpage_{timestamp}.md
```

Example:
```
data/documents/goudse_expat_pakket/webpage_20260120.md
```